In [9]:
import os
import pandas as pd
from pathlib import Path

# Point this at your dataset root
BASE = Path.home() / "raga-multimodal-classifier" / "data" / \
       "RaagaDhvani-A novel curated Augmented Carnatic mus" / \
       "RaagaDhvani_Dataset_VidushiArchana"

# Recursively collect every .wav file
records = []
for wav_path in BASE.rglob("*.wav"):
    records.append({
        "filename": wav_path.name,
        "folder": wav_path.parent.name,
        "full_path": str(wav_path)
    })

df = pd.DataFrame(records)
print(f"Total .wav files found: {len(df)}")
print(f"\nFiles per folder:")
print(df["folder"].value_counts())
print(f"\nFirst few filenames:")
for name in df["filename"].head(10):
    print(" ", name)

Total .wav files found: 613

Files per folder:
folder
Data_Augmentation     448
Studio_Audio_split    165
Name: count, dtype: int64

First few filenames:
  Hamsadhwani_happy_09.wav
  Bhairavi_devotion_14.wav
  Poorvikalyani_12.wav
  Poorvikalyani_06.wav
  Thodi_09.wav
  Gaanamurthe_devotion_06.wav
  Gaanamurthe_devotion_12.wav
  Gaanamurthe_devotion_13.wav
  Gaanamurthe_devotion_07.wav
  Thodi_08.wav


In [10]:
readme_path = BASE / "Readme.txt"
print(readme_path.read_text())

# Carnatic Raga Emotion Audio Dataset-RaagaDhvani

## Description:
This dataset comprises original instrumental and vocal recordings of select Carnatic ragas, annotated with corresponding emotional labels derived from expert musicological analysis and listener feedback. The dataset is designed to support research in emotion recognition, computational musicology, cultural music studies, and AI-based music applications.

## Contents:
- Audio Files: High-quality WAV recordings of individual ragas
- Emotion Labels: Mapped emotions per raga (e.g., tranquility, devotion, blissfulness)
- Metadata: Includes raga name, recording conditions, and emotion annotations

## License:
Carnatic Raga Emotion Audio Dataset- RaagaDhvani © 2025 by Vidushi .Mrs.Archana Priyadarshini is licensed under CC BY-ND 4.0. 

You are free to:
- Use and share this dataset for research, teaching, and non-commercial academic purposes
- Redistribute without alteration, with credit to the creator

You are **not permitted**

In [16]:
import re

# All 11 canonical ragas, lowercased for matching
RAGA_ALIASES = {
    "anandabhairavi": ["anandabhairavi"], 
    "poorvikalyani": ["poorvikalyani"], 
    "hamsadhwani": ["hamsadhwani", "hamsadwani"], 
    "gaanamurthe": ["gaanamurthe", "ganamurthe"],
    "dwijavanthi": ["dwijavanthi", "dhwijavanthi"], 
    "neelambari": ["neelambari", "nilambari"], 
    "bhairavi": ["bhairavi"], 
    "bilahari": ["bilahari"],
    "saveri": ["saveri"], 
    "bhauli": ["bhauli", "bowli", "bhowli"], 
    "thodi": ["thodi"]
}

ALIAS_MAP = sorted(
    [(a, canon) for canon, al in RAGA_ALIASES.items() for a in al],
    key=lambda x: len(x[0]), reverse=True
)


def parse_filename(fname):
    s = fname.replace(".wav", "").lower().strip()
    s = re.sub(r"^f_?(?:flute_?)?", "", s)                       # drop flute prefix

    raga = None
    for alias, canon in ALIAS_MAP:
        if s.startswith(alias):
            raga = canon
            s = s[len(alias):]
            break
    s = re.sub(r"^(raaga|raga)_?", "", s)           # drop 'raaga' suffix

    aug_type, aug_param = "original", None
    for a in ["pitch_shift", "time_stretch", "noise"]:
        m = re.search(a + r"_(-?\d+(?:\.\d+)?)", s)
        if m:
            aug_type, aug_param = a, float(m.group(1))
            s = s[:m.start()].rstrip("_")
            break

    s = re.sub(r"_?\d+$", "", s).strip("_")          # drop trailing index
    emotion = s if s else None

    return pd.Series({"raga": raga, "emotion": emotion,
                      "aug_type": aug_type, "aug_param": aug_param})

# Apply parser
df = df[["filename", "folder", "full_path"]].copy()
df = df.join(df["filename"].apply(parse_filename))

# Report parse failures
failed = df[df["raga"].isna()]
print(f"Unparsed files: {len(failed)}")
if len(failed): print(failed["filename"].tolist()[:20])

# Build canonical raga -> emotion map from labeled rows, then fill gaps
raga_emotion = (df.dropna(subset=["emotion"])
                  .groupby("raga")["emotion"].agg(lambda x: x.mode()[0]))
df["emotion"] = df["emotion"].fillna(df["raga"].map(raga_emotion))

print("\nRaga -> emotion map:"); print(raga_emotion)
print("\nFiles per raga:"); print(df["raga"].value_counts())
print("\nOriginal vs augmented:"); print(df["aug_type"].value_counts())

Unparsed files: 0

Raga -> emotion map:
raga
anandabhairavi        blissfulness
bhairavi                  devotion
bhauli                  melanchony
bilahari               tranquility
dwijavanthi                   calm
gaanamurthe               devotion
hamsadhwani                  happy
neelambari        relaxation_peace
poorvikalyani             serenity
saveri                  compassion
thodi                          sad
Name: emotion, dtype: object

Files per raga:
raga
hamsadhwani       141
saveri            131
poorvikalyani      51
neelambari         51
thodi              42
bhairavi           33
gaanamurthe        33
dwijavanthi        33
anandabhairavi     33
bhauli             33
bilahari           32
Name: count, dtype: int64

Original vs augmented:
aug_type
original        165
time_stretch    150
pitch_shift     149
noise           149
Name: count, dtype: int64


In [17]:
# Structure: how originals vs augmentations distribute per raga
print("Raga x augmentation type:")
print(pd.crosstab(df["raga"], df["aug_type"]))

# Lock emotion as a clean, deterministic function of raga
RAGA_EMOTION = {
    "anandabhairavi": "blissfulness", "bhairavi": "devotion",
    "bhauli": "melancholy", "bilahari": "tranquility", "dwijavanthi": "calm",
    "gaanamurthe": "devotion", "hamsadhwani": "happy", "neelambari": "relaxation",
    "poorvikalyani": "serenity", "saveri": "compassion", "thodi": "sad",
}
df["emotion"] = df["raga"].map(RAGA_EMOTION)

df.to_csv("metadata_full.csv", index=False)
print("\nSaved metadata_full.csv")

Raga x augmentation type:
aug_type        noise  original  pitch_shift  time_stretch
raga                                                      
anandabhairavi      6        15            6             6
bhairavi            6        15            6             6
bhauli              6        15            6             6
bilahari            6        15            5             6
dwijavanthi         6        15            6             6
gaanamurthe         6        15            6             6
hamsadhwani        42        15           42            42
neelambari         12        15           12            12
poorvikalyani      12        15           12            12
saveri             38        15           39            39
thodi               9        15            9             9

Saved metadata_full.csv


In [18]:
from sklearn.model_selection import train_test_split

originals = df[df["aug_type"] == "original"].copy()
print(f"Original recordings: {len(originals)}\n")
print("Originals per raga:")
print(originals["raga"].value_counts())

train_df, test_df = train_test_split(
    originals, test_size=0.2,
    stratify=originals["raga"], random_state=42
)
print(f"\nTrain: {len(train_df)}   Test: {len(test_df)}")

train_df.to_csv("train_originals.csv", index=False)
test_df.to_csv("test_originals.csv", index=False)
print("Saved train_originals.csv and test_originals.csv")

Original recordings: 165

Originals per raga:
raga
hamsadhwani       15
bhairavi          15
poorvikalyani     15
thodi             15
gaanamurthe       15
dwijavanthi       15
neelambari        15
bilahari          15
anandabhairavi    15
bhauli            15
saveri            15
Name: count, dtype: int64

Train: 132   Test: 33
Saved train_originals.csv and test_originals.csv
